In [1]:
# region Imports

#* --------------------------------------------------------------------------------
#* General purpose imports
#* --------------------------------------------------------------------------------
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import fisher_exact, barnard_exact
from matplotlib.ticker import FuncFormatter
import pickle as pkl


#* --------------------------------------------------------------------------------
#* Personal librairies imports
#* --------------------------------------------------------------------------------
import sys, os
src_path = os.path.abspath(os.path.join("..", "src"))
if src_path not in sys.path:
    sys.path.insert(0, src_path)
from utils import astro_utils as au
from utils import maths_utils  as mu
from utils import stats_utils  as su
from utils import graphics_utils  as gu
from utils import labels_utils  as lu
from utils import pandas_utils  as pu


#* --------------------------------------------------------------------------------
#* Project modules imports
#* --------------------------------------------------------------------------------
import sSFR
import generate_report as report
import domination

#* --------------------------------------------------------------------------------
#* Global variables
#* --------------------------------------------------------------------------------
import config as co

#* --------------------------------------------------------------------------------
#* Project data
#* --------------------------------------------------------------------------------

with open(co.DATA_PATH + co.PROCESS_SAMPLES, "rb") as file:
            sample = pkl.load(file)



# endregion

Done


In [2]:
sample['CG4'+co.GASUFF].columns

Index(['objid', 'specobjid', 'Group', 'RA', 'Dec', 'M_r', 'Lum', 'z',
       'dist2BGG', 'lgm', 'sfr', 'sSFR', 'rank_dist', 'rank_M', 'RA_BGG',
       'Dec_BGG', 'M_BGG', 'sSFR_status', 'p_E', 'p_S', 'is_dominated',
       'morphology', 'sSFR_raw', 'sSFR_MS_offset', 'MS_res', 'sSFR_excess'],
      dtype='object')

In [3]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import mannwhitneyu

GROUP_COL = "Group"
RANK_COL  = "rank_M"   # BGG si == 1
MASS_COL  = "lgm"      # log10(M*) ou équivalent

def clean_morph(s: pd.Series) -> pd.Series:
    """Garde seulement Spiral/Elliptical, met le reste à NaN (drop ensuite)."""
    return s.where(s.isin(["Spiral", "Elliptical"]))

def prep_df(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["morph_clean"] = clean_morph(df["morphology"])
    df["_is_bgg"] = df[RANK_COL].eq(1)
    # garde au moins les colonnes utiles
    return df

def group_level_binom(df: pd.DataFrame, control_lgm_bgg: bool = True):
    """
    Analyse au niveau groupe :
      k_sp_sat ~ Binomial(n_sat, p), logit(p) = b0 + b1*(BGG is Spiral) + b2*lgm_BGG(option)
    """
    df = prep_df(df)
    df = df.dropna(subset=[GROUP_COL, "morph_clean"])

    # BGG morph & masse par groupe
    bgg = df[df["_is_bgg"]].loc[:, [GROUP_COL, "morph_clean", MASS_COL]].dropna(subset=[GROUP_COL, "morph_clean"])
    bgg = bgg.rename(columns={"morph_clean": "bgg_morph", MASS_COL: "lgm_bgg"})

    # satellites only
    sat = df[~df["_is_bgg"]].copy()

    # stats satellites par groupe (en excluant Uncertain/NaN déjà)
    g = sat.groupby(GROUP_COL)
    grp = pd.DataFrame({
        "n_sat": g.size(),
        "k_sp_sat": g["morph_clean"].apply(lambda x: (x == "Spiral").sum())
    }).reset_index()

    grp = grp.merge(bgg, on=GROUP_COL, how="inner")
    grp = grp[grp["n_sat"] > 0].copy()

    grp["f_sp_sat"] = grp["k_sp_sat"] / grp["n_sat"]
    grp["bgg_is_sp"] = (grp["bgg_morph"] == "Spiral").astype(int)

    # GLM binomial pondéré (var_weights = n_sat)
    Xcols = ["bgg_is_sp"]
    if control_lgm_bgg and "lgm_bgg" in grp.columns:
        Xcols.append("lgm_bgg")

    X = sm.add_constant(grp[Xcols])
    y = grp["f_sp_sat"]
    w = grp["n_sat"]

    model = sm.GLM(y, X, family=sm.families.Binomial(), var_weights=w)
    res = model.fit()

    # Effet principal : bgg_is_sp
    beta = res.params["bgg_is_sp"]
    se   = res.bse["bgg_is_sp"]
    OR   = float(np.exp(beta))
    CI95 = (float(np.exp(beta - 1.96*se)), float(np.exp(beta + 1.96*se)))
    pval = float(res.pvalues["bgg_is_sp"])

    # Résumé médianes + MWU sur f_sp_sat
    a = grp.loc[grp["bgg_is_sp"]==1, "f_sp_sat"]
    b = grp.loc[grp["bgg_is_sp"]==0, "f_sp_sat"]
    p_mwu = float(mannwhitneyu(a, b, alternative="two-sided").pvalue) if (len(a)>0 and len(b)>0) else np.nan

    summary = {
        "n_groups_used": int(len(grp)),
        "median_fSp_sat_if_BGG_Sp": float(a.median()) if len(a) else np.nan,
        "median_fSp_sat_if_BGG_Ell": float(b.median()) if len(b) else np.nan,
        "OR_BGGSp_effect": OR,
        "CI95_OR": CI95,
        "p_glm": pval,
        "p_mwu": p_mwu,
        "controls": Xcols[1:]  # sans const
    }
    return summary, grp, res

def satellite_level_cluster(df: pd.DataFrame,
                            control_lgm_sat: bool = False,
                            control_lgm_bgg: bool = True):
    """
    Analyse au niveau satellites (1 ligne = 1 satellite),
    logit(P(Spiral_sat)) = a0 + a1*(BGG is Spiral) + covariables...
    SE clusterisées par Group.
    """
    df = prep_df(df)
    df = df.dropna(subset=[GROUP_COL, "morph_clean"])

    # BGG morph + lgm_bgg par groupe
    bgg = df[df["_is_bgg"]].loc[:, [GROUP_COL, "morph_clean", MASS_COL]].dropna(subset=[GROUP_COL, "morph_clean"])
    bgg = bgg.rename(columns={"morph_clean": "bgg_morph", MASS_COL: "lgm_bgg"})

    sat = df[~df["_is_bgg"]].copy()
    sat = sat.merge(bgg, on=GROUP_COL, how="inner")
    sat = sat.dropna(subset=["bgg_morph", "morph_clean"])

    sat["y_sp"] = (sat["morph_clean"] == "Spiral").astype(int)
    sat["bgg_is_sp"] = (sat["bgg_morph"] == "Spiral").astype(int)

    Xcols = ["bgg_is_sp"]
    if control_lgm_bgg and "lgm_bgg" in sat.columns:
        Xcols.append("lgm_bgg")
    if control_lgm_sat and MASS_COL in sat.columns:
        Xcols.append(MASS_COL)

    X = sm.add_constant(sat[Xcols])
    y = sat["y_sp"]

    model = sm.GLM(y, X, family=sm.families.Binomial())
    res = model.fit(cov_type="cluster", cov_kwds={"groups": sat[GROUP_COL]})

    beta = res.params["bgg_is_sp"]
    se   = res.bse["bgg_is_sp"]
    OR   = float(np.exp(beta))
    CI95 = (float(np.exp(beta - 1.96*se)), float(np.exp(beta + 1.96*se)))
    pval = float(res.pvalues["bgg_is_sp"])

    summary = {
        "n_sat_used": int(len(sat)),
        "n_groups_used": int(sat[GROUP_COL].nunique()),
        "OR_BGGSp_effect": OR,
        "CI95_OR": CI95,
        "p_cluster": pval,
        "controls": Xcols[1:]  # sans const
    }
    return summary, sat, res

# -----------------------------
# Boucle sur tes catégories
# -----------------------------
all_results = {}

for cat in co.SAMPLE.keys():
    key = cat + co.GASUFF
    if key not in sample:
        continue

    df = sample[key].copy()

    # garde seulement si les colonnes existent
    needed = {GROUP_COL, RANK_COL, "morphology", MASS_COL}
    if not needed.issubset(df.columns):
        missing = sorted(list(needed - set(df.columns)))
        all_results[cat] = {"error": f"Colonnes manquantes: {missing}"}
        continue

    # Groupe-level: contrôle lgm_BGG par défaut
    out_g, grp, res_g = group_level_binom(df, control_lgm_bgg=True)

    # Satellite-level: cluster par groupe, contrôle lgm_BGG, option lgm_sat
    out_s, sat, res_s = satellite_level_cluster(df, control_lgm_sat=False, control_lgm_bgg=True)

    all_results[cat] = {
        "group_level": out_g,
        "satellite_level_cluster": out_s,
    }

all_results

{'CG4': {'group_level': {'n_groups_used': 52,
   'median_fSp_sat_if_BGG_Sp': 0.3333333333333333,
   'median_fSp_sat_if_BGG_Ell': 0.3333333333333333,
   'OR_BGGSp_effect': 0.8900790967269001,
   'CI95_OR': (0.42277810248658604, 1.8738926963591038),
   'p_glm': 0.759168528700144,
   'p_mwu': 0.9839023840860037,
   'controls': ['lgm_bgg']},
  'satellite_level_cluster': {'n_sat_used': 140,
   'n_groups_used': 52,
   'OR_BGGSp_effect': 0.890079096730997,
   'CI95_OR': (0.4166969807948871, 1.9012395936399558),
   'p_cluster': 0.7636276741671504,
   'controls': ['lgm_bgg']}},
 'Control4B': {'group_level': {'n_groups_used': 597,
   'median_fSp_sat_if_BGG_Sp': 0.6666666666666666,
   'median_fSp_sat_if_BGG_Ell': 0.6666666666666666,
   'OR_BGGSp_effect': 1.1116417716387064,
   'CI95_OR': (0.8850242619178382, 1.3962864992810367),
   'p_glm': 0.3628634655472326,
   'p_mwu': 0.1230468342728902,
   'controls': ['lgm_bgg']},
  'satellite_level_cluster': {'n_sat_used': 1581,
   'n_groups_used': 597,
  

Refaire en séparant Dom et Non Dom

In [4]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import mannwhitneyu

# -----------------------------
# Config colonnes
# -----------------------------
GROUP_COL = "Group"
RANK_COL  = "rank_M"       # BGG si == 1
MASS_COL  = "lgm"          # log masse
DOM_COL   = "is_dominated" # dans la table GROUPES (cat + co.GRSUFF)

# -----------------------------
# Helpers
# -----------------------------
def clean_morph(s: pd.Series) -> pd.Series:
    """Garde seulement Spiral/Elliptical, le reste -> NaN (Uncertain, NaN)."""
    return s.where(s.isin(["Spiral", "Elliptical"]))

def attach_dom_from_group_table(df_gal: pd.DataFrame, df_grp: pd.DataFrame) -> pd.DataFrame:
    """
    IMPORTANT:
    - Cherche DOM_COL uniquement dans df_grp = sample[cat + co.GRSUFF]
    - Ajoute/écrase DOM_COL dans df_gal par mapping Group -> is_dominated
    """
    out = df_gal.copy()

    # mapping Group -> is_dominated (bool)
    dom_map = (
        df_grp[[GROUP_COL, DOM_COL]]
        .drop_duplicates(subset=[GROUP_COL])
        .set_index(GROUP_COL)[DOM_COL]
    )

    out[DOM_COL] = out[GROUP_COL].map(dom_map).astype("boolean")  # bool pandas
    return out

def prep_df(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["morph_clean"] = clean_morph(df["morphology"])
    df["_is_bgg"] = df[RANK_COL].eq(1)
    return df

# -----------------------------
# Stats : niveau groupe (binomial pondéré)
# -----------------------------
def group_level_binom(df: pd.DataFrame, control_lgm_bgg: bool = True):
    df = prep_df(df)
    df = df.dropna(subset=[GROUP_COL, "morph_clean"])

    # BGG morph + masse par groupe
    bgg = df[df["_is_bgg"]][[GROUP_COL, "morph_clean", MASS_COL]].dropna(subset=[GROUP_COL, "morph_clean"])
    bgg = bgg.rename(columns={"morph_clean": "bgg_morph", MASS_COL: "lgm_bgg"})

    # satellites
    sat = df[~df["_is_bgg"]].copy()

    # stats satellites par groupe
    g = sat.groupby(GROUP_COL)
    grp = pd.DataFrame({
        "n_sat": g.size(),
        "k_sp_sat": g["morph_clean"].apply(lambda x: (x == "Spiral").sum())
    }).reset_index()

    grp = grp.merge(bgg, on=GROUP_COL, how="inner")
    grp = grp[grp["n_sat"] > 0].copy()

    grp["f_sp_sat"] = grp["k_sp_sat"] / grp["n_sat"]
    grp["bgg_is_sp"] = (grp["bgg_morph"] == "Spiral").astype(int)

    # GLM binomial pondéré : y=f, var_weights=n_sat
    Xcols = ["bgg_is_sp"]
    if control_lgm_bgg and "lgm_bgg" in grp.columns:
        Xcols.append("lgm_bgg")

    X = sm.add_constant(grp[Xcols])
    y = grp["f_sp_sat"]
    w = grp["n_sat"]

    res = sm.GLM(y, X, family=sm.families.Binomial(), var_weights=w).fit()

    beta = res.params["bgg_is_sp"]
    se   = res.bse["bgg_is_sp"]
    OR   = float(np.exp(beta))
    CI95 = (float(np.exp(beta - 1.96*se)), float(np.exp(beta + 1.96*se)))
    pval = float(res.pvalues["bgg_is_sp"])

    # MWU sur f_sp_sat (optionnel)
    a = grp.loc[grp["bgg_is_sp"]==1, "f_sp_sat"]
    b = grp.loc[grp["bgg_is_sp"]==0, "f_sp_sat"]
    p_mwu = float(mannwhitneyu(a, b, alternative="two-sided").pvalue) if (len(a)>0 and len(b)>0) else np.nan

    return {
        "n_groups_used": int(len(grp)),
        "median_fSp_sat_BGGSp": float(a.median()) if len(a) else np.nan,
        "median_fSp_sat_BGGEll": float(b.median()) if len(b) else np.nan,
        "OR": OR,
        "CI95_low": CI95[0],
        "CI95_high": CI95[1],
        "p_glm": pval,
        "p_mwu": p_mwu,
    }

# -----------------------------
# Stats : niveau satellite, SE clusterisées par Group
# -----------------------------
def satellite_level_cluster(df: pd.DataFrame, control_lgm_bgg: bool = True, control_lgm_sat: bool = False):
    df = prep_df(df)
    df = df.dropna(subset=[GROUP_COL, "morph_clean"])

    # BGG morph + masse par groupe
    bgg = df[df["_is_bgg"]][[GROUP_COL, "morph_clean", MASS_COL]].dropna(subset=[GROUP_COL, "morph_clean"])
    bgg = bgg.rename(columns={"morph_clean": "bgg_morph", MASS_COL: "lgm_bgg"})

    sat = df[~df["_is_bgg"]].merge(bgg, on=GROUP_COL, how="inner")
    sat = sat.dropna(subset=["bgg_morph", "morph_clean"])

    sat["y_sp"] = (sat["morph_clean"] == "Spiral").astype(int)
    sat["bgg_is_sp"] = (sat["bgg_morph"] == "Spiral").astype(int)

    Xcols = ["bgg_is_sp"]
    if control_lgm_bgg and "lgm_bgg" in sat.columns:
        Xcols.append("lgm_bgg")
    if control_lgm_sat and MASS_COL in sat.columns:
        Xcols.append(MASS_COL)

    X = sm.add_constant(sat[Xcols])
    y = sat["y_sp"]

    res = sm.GLM(y, X, family=sm.families.Binomial()).fit(
        cov_type="cluster",
        cov_kwds={"groups": sat[GROUP_COL]}
    )

    beta = res.params["bgg_is_sp"]
    se   = res.bse["bgg_is_sp"]
    OR   = float(np.exp(beta))
    CI95 = (float(np.exp(beta - 1.96*se)), float(np.exp(beta + 1.96*se)))
    pval = float(res.pvalues["bgg_is_sp"])

    return {
        "n_sat_used": int(len(sat)),
        "n_groups_used": int(sat[GROUP_COL].nunique()),
        "OR": OR,
        "CI95_low": CI95[0],
        "CI95_high": CI95[1],
        "p_cluster": pval,
    }

# -----------------------------
# Boucle: cat, split dominated/non-dominated
# IMPORTANT: is_dominated lu dans cat+co.GRSUFF uniquement
# -----------------------------
rows = []

for cat in co.SAMPLE.keys():
    key_gal = cat + co.GASUFF   # galaxies
    key_grp = cat + co.GRSUFF   # groupes (is_dominated)

    if key_gal not in sample or key_grp not in sample:
        continue

    df_gal = sample[key_gal].copy()
    df_grp = sample[key_grp].copy()

    # Vérifs colonnes
    needed_gal = {GROUP_COL, RANK_COL, "morphology", MASS_COL}
    needed_grp = {GROUP_COL, DOM_COL}

    if not needed_gal.issubset(df_gal.columns):
        rows.append({"cat": cat, "dom": "ALL", "error": f"Missing in GAL: {sorted(list(needed_gal-set(df_gal.columns)))}"})
        continue
    if not needed_grp.issubset(df_grp.columns):
        rows.append({"cat": cat, "dom": "ALL", "error": f"Missing in GRP: {sorted(list(needed_grp-set(df_grp.columns)))}"})
        continue

    # Attache is_dominated depuis la table GROUPES
    df = attach_dom_from_group_table(df_gal, df_grp)

    # Split
    for dom_value, dom_label in [(True, "dominated"), (False, "non_dominated")]:
        dfi = df[df[DOM_COL] == dom_value].copy()

        # si trop peu de groupes, skip
        nG = dfi[GROUP_COL].nunique()
        if nG < 10:
            rows.append({"cat": cat, "dom": dom_label, "warning": f"too_few_groups ({nG})"})
            continue

        out_g = group_level_binom(dfi, control_lgm_bgg=True)
        out_s = satellite_level_cluster(dfi, control_lgm_bgg=True, control_lgm_sat=False)

        rows.append({
            "cat": cat,
            "dom": dom_label,

            "G_n_groups": out_g["n_groups_used"],
            "G_med_fSp_sat_BGGSp": out_g["median_fSp_sat_BGGSp"],
            "G_med_fSp_sat_BGGEll": out_g["median_fSp_sat_BGGEll"],
            "G_OR": out_g["OR"],
            "G_CI95": f"[{out_g['CI95_low']:.2f}, {out_g['CI95_high']:.2f}]",
            "G_p": out_g["p_glm"],
            "G_p_MWU": out_g["p_mwu"],

            "S_n_sat": out_s["n_sat_used"],
            "S_n_groups": out_s["n_groups_used"],
            "S_OR": out_s["OR"],
            "S_CI95": f"[{out_s['CI95_low']:.2f}, {out_s['CI95_high']:.2f}]",
            "S_p": out_s["p_cluster"],
        })

results_df = pd.DataFrame(rows)

# Mise en forme pour lecture
if len(results_df):
    results_df = results_df.sort_values(["cat", "dom"])
results_df

,cat,dom,G_n_groups,G_med_fSp_sat_BGGSp,G_med_fSp_sat_BGGEll,G_OR,G_CI95,G_p,G_p_MWU,S_n_sat,S_n_groups,S_OR,S_CI95,S_p
0,CG4,dominated,32,0.666667,0.333333,1.274640,"[0.43, 3.78]",0.661964,0.216704,88,32,1.274640,"[0.56, 2.90]",0.562974
1,CG4,non_dominated,20,0.333333,0.583333,0.533493,"[0.18, 1.61]",0.264298,0.229406,52,20,0.533493,"[0.16, 1.74]",0.296560
2,Control4B,dominated,165,1.000000,0.666667,1.841810,"[1.15, 2.95]",0.011147,0.009792,437,165,1.841810,"[1.12, 3.04]",0.016585
3,Control4B,non_dominated,432,0.666667,0.666667,0.929992,"[0.71, 1.21]",0.592947,0.907025,1144,432,0.929992,"[0.71, 1.22]",0.600729
4,Control4C,dominated,452,0.666667,0.500000,1.840049,"[1.39, 2.44]",0.000021,0.000001,1149,452,1.840049,"[1.38, 2.45]",0.000027
5,Control4C,non_dominated,182,0.666667,0.666667,0.888962,"[0.60, 1.33]",0.564258,0.975554,467,182,0.888962,"[0.57, 1.39]",0.604472
6,RG4,dominated,22,1.000000,0.666667,1.360954,"[0.34, 5.38]",0.660350,0.293773,57,22,1.360954,"[0.46, 4.02]",0.576872
7,RG4,non_dominated,25,0.666667,0.666667,0.593942,"[0.21, 1.68]",0.326667,0.162955,68,25,0.593942,"[0.25, 1.43]",0.243828


In [5]:
results_df.to_csv(co.OUTPUT_PATH + "morphology_by_domination.csv", index=False)

Fisher exact test p-value: 0.0260
